In [6]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, models
import os

In [14]:
def original_model(input_shape=(224,224,3),num_classes=10):
  inputs = layers.Input(shape=input_shape)

  # Block 1
  x = layers.Conv2D(16, (7, 7), padding="same", activation="relu")(inputs)
  x = layers.Conv2D(16, (7, 7), padding="same", activation="relu")(x)
  x = layers.AveragePooling2D((2, 2))(x)

  # Block 2
  x = layers.Conv2D(32, (5, 5), padding="same", activation="relu")(x)
  x = layers.Conv2D(32, (5, 5), padding="same", activation="relu")(x)
  x = layers.AveragePooling2D((2, 2))(x)

  # Bloack 3
  x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
  x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
  x = layers.AveragePooling2D((2, 2))(x)

  # Block 4
  x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
  x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
  x = layers.AveragePooling2D((2, 2))(x)

  # conv 9 & 10
  x = layers.Conv2D(256, (3, 3), padding="same", activation="relu")(x)
  x = layers.Conv2D(num_classes, (3, 3), padding="same")(x)

  # Global pooling layer
  x = layers.GlobalAveragePooling2D()(x)

  # softmax
  outputs = layers.Softmax()(x)

  model = models.Model(inputs, outputs, name="Original_CNN")
  return model


In [15]:
model1 = original_model()
model1.summary()

Model: "Original_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 224, 224, 16)   │         2,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 224, 224, 16)   │        12,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_4             │ (None, 112, 112, 16)   │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 112, 112, 32)   │        12,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 112, 112, 32)   │        25,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_5             │ (None, 56, 56, 32)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 56, 56, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 56, 56, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_6             │ (None, 28, 28, 64)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 28, 28, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_7             │ (None, 14, 14, 128)    │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 14, 14, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 14, 14, 10)     │        23,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 10)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Softmax)               │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 648,474 (2.47 MB)

 Trainable params: 648,474 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
PROCESSED_DIR = "../data/processed"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
CLASS_NAMES = ["c0","c1","c2","c3","c4","c5","c6","c7","c8","c9"]

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(PROCESSED_DIR, "train"),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(PROCESSED_DIR, "val"),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(PROCESSED_DIR, "test"),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

Found 13451 files belonging to 10 classes.
Found 4483 files belonging to 10 classes.
Found 4490 files belonging to 10 classes.


In [17]:
normalization = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization(x), y))
val_ds   = val_ds.map(lambda x, y: (normalization(x), y))
test_ds  = test_ds.map(lambda x, y: (normalization(x), y))

In [18]:
for images, labels in train_ds.take(1):
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)

Images shape: (16, 224, 224, 3)
Labels shape: (16, 10)


In [19]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

model1.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=200
)

Epoch 1/200
841/841 ━━━━━━━━━━━━━━━━━━━━ 344s 406ms/step - accuracy: 0.1117 - loss: 2.3011 - val_accuracy: 0.1111 - val_loss: 2.2942
Epoch 2/200
841/841 ━━━━━━━━━━━━━━━━━━━━ 346s 411ms/step - accuracy: 0.2792 - loss: 1.8556 - val_accuracy: 0.3928 - val_loss: 1.5117
Epoch 3/200
841/841 ━━━━━━━━━━━━━━━━━━━━ 371s 397ms/step - accuracy: 0.4664 - loss: 1.4024 - val_accuracy: 0.5412 - val_loss: 1.2088
Epoch 4/200
841/841 ━━━━━━━━━━━━━━━━━━━━ 331s 394ms/step - accuracy: 0.6017 - loss: 1.1066 - val_accuracy: 0.6527 - val_loss: 0.9810
Epoch 5/200
841/841 ━━━━━━━━━━━━━━━━━━━━ 346s 411ms/step - accuracy: 0.7271 - loss: 0.8125 - val_accuracy: 0.8024 - val_loss: 0.6219
Epoch 6/200
127/841 ━━━━━━━━━━━━━━━━━━━━ 4:25 371ms/step - accuracy: 0.7940 - loss: 0.6361

KeyboardInterrupt: 

In [ ]:
model1.save("../models/original_cnn.h5")

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['Train', 'Validation'])
plt.title('Accuracy')
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['Train', 'Validation'])
plt.title('Loss')
plt.show()

In [ ]:
test_loss, test_acc = model1.evaluate(test_ds)
print("Test Accuracy:", test_acc)

In [ ]:
import time
import numpy as np

sample_batch = next(iter(test_ds))[0]

start = time.time()
pred = model1.predict(sample_batch)
end = time.time()

time_per_image = (end - start) / len(sample_batch)
print("Time per image (seconds):", time_per_image)